# CS781 Phase 1 — Legal LM from scratch (JAX, Kaggle TPU v3-8)
Byte-level BPE (A1-style, rewritten for speed) → hand-written JAX models: attention decoders AND attention-free gated linear-recurrent / hybrid recurrent+attention models (ablation) → time-budgeted final run → sweep decoding → `submission.zip`.

## 1. Setup and configuration

In [1]:
import os, sys, time, json, math, random, pickle, subprocess, zipfile, gc, heapq, importlib.util
from collections import Counter, defaultdict
T0 = time.time()                      # wall-clock start of the whole notebook (9h Kaggle limit)

def _pip(pkg): subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
for _m, _p in [("regex", "regex"), ("bert_score", "bert-score"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(_m) is None: _pip(_p)

import numpy as np
import regex as re2
from huggingface_hub import hf_hub_download, HfApi, login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
    login(token=HF_TOKEN)
except Exception as e:
    HF_TOKEN = None
    print("HF login skipped:", e)
# ---------------- configuration ----------------
SEED = 42; random.seed(SEED); np.random.seed(SEED); rng = random.Random(SEED)
HF_REPO = "Exploration-Lab/CS-781-Capstone"
WORK = "/kaggle/working/cs781_phase1"; TMP = "/tmp/hf"
os.makedirs(WORK, exist_ok=True); os.makedirs(TMP, exist_ok=True)

TOTAL_SEC   = 8.6 * 3600      # hard budget for the whole notebook (leaves ~24 min slack under 9h)
RESERVE_SEC = 60 * 60         # kept free after training for decoding sweep + test generation + scoring
FINAL_MAX_MIN = 330           # upper cap for the final training run (auto-shrunk by the budget guard)

# data mixing (share of the character budget per source) and sampling
MIX = {"acts": 0.05, "high_courts": 0.35, "supreme_court": 0.35, "tribunals": 0.25}
TOTAL_CHARS = 3.5e9           # ~ 800M tokens with a 16k byte-level BPE
VAL_P, VAL_CHARS_PER_SRC = 0.003, 4_000_000
TOK_CHARS = 80_000_000        # text sample used to train the tokenizer
VOCAB = 16384

# model / sequence
CTX, PROMPT_MAX, GEN_TOK = 1280, 960, 320      # prompt (512 words) is left-truncated to PROMPT_MAX tokens
D, L, H, HD, V = 768, 12, 12, 64, VOCAB
PCB = 8                                         # sequences per TPU core per step
PEAK_LR, MIN_LR_FRAC, WD, B1, B2, EPS, CLIP = 8e-4, 0.1, 0.1, 0.9, 0.95, 1e-8, 1.0
ABL_STEPS = 300                                 # steps per architecture in the ablation
TARGET_WORDS = 128; N_VAL_PROMPTS = 96
print("config ok")


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


config ok


## 2. Data: stream official Phase-1 files, sample by mixing policy, hold out validation docs

In [2]:
def iter_json_array(path, chunk=1 << 26):
    """Stream the objects of a top-level JSON array without loading the whole file."""
    dec = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        buf = f.read(chunk); i = 0
        while i < len(buf) and buf[i] in " \n\r\t": i += 1
        i += 1                                            # skip '['
        while True:
            while True:
                while i < len(buf) and buf[i] in " \n\r\t,": i += 1
                if i < len(buf): break
                more = f.read(chunk)
                if not more: return
                buf, i = more, 0
            if buf[i] == "]": return
            try:
                obj, j = dec.raw_decode(buf, i); i = j; yield obj
            except json.JSONDecodeError:
                more = f.read(chunk)
                if not more: return
                buf, i = buf[i:] + more, 0

TRAIN_NPY, VAL_NPY, VALDOC_JSON, TOK_PATH = f"{WORK}/train_tokens.npy", f"{WORK}/val_tokens.npy", f"{WORK}/val_docs.json", f"{WORK}/bpe_legal.json"
CACHED = all(os.path.exists(x) for x in [TRAIN_NPY, VAL_NPY, VALDOC_JSON, TOK_PATH])
api = HfApi()
files = api.list_repo_files(HF_REPO, repo_type="dataset", token=HF_TOKEN)
tr_files = {k: next(f for f in files if f.startswith("Phase_1_Pretraining/Train/") and k in f) for k in MIX}
test_file = next(f for f in files if f.startswith("Phase_1_Pretraining/Test/") and f.endswith(".json"))
print(tr_files, test_file)

train_docs, train_src, val_docs = [], [], []
stats = {}
for src, fn in ([] if CACHED else tr_files.items()):
    path = hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN, local_dir=TMP)
    size = os.path.getsize(path); budget = MIX[src] * TOTAL_CHARS
    p = min(1.0, 1.2 * budget / size)                     # uniform subsample across the file
    got = vch = seen = 0
    for rec in iter_json_array(path):
        seen += 1
        t = " ".join(rec["text"].split())                 # whitespace-normalised (test prompts are space-joined words)
        if len(t) < 200: continue
        u = rng.random()
        if u < VAL_P:
            if vch < VAL_CHARS_PER_SRC: val_docs.append((src, t)); vch += len(t)
        elif u < VAL_P + p:
            train_docs.append(t); train_src.append(src); got += len(t)
            if got >= budget: break
    stats[src] = dict(file_MB=size / 1e6, docs_scanned=seen, train_chars=got, val_chars=vch)
    print(src, stats[src]); os.remove(path); gc.collect()
if CACHED: val_docs = json.load(open(VALDOC_JSON, encoding="utf-8"))
else: json.dump(val_docs, open(VALDOC_JSON, "w", encoding="utf-8")); json.dump(stats, open(f"{WORK}/data_stats.json", "w"), indent=1)
print("train docs:", len(train_docs), "val docs:", len(val_docs))

{'acts': 'Phase_1_Pretraining/Train/train_acts.json', 'high_courts': 'Phase_1_Pretraining/Train/train_high_courts.json', 'supreme_court': 'Phase_1_Pretraining/Train/train_supreme_court.json', 'tribunals': 'Phase_1_Pretraining/Train/train_tribunals.json'} Phase_1_Pretraining/Test/test.json


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

acts {'file_MB': 728.801891, 'docs_scanned': 16124, 'train_chars': 175101383, 'val_chars': 2837180}


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

high_courts {'file_MB': 10672.497795, 'docs_scanned': 2316367, 'train_chars': 1225015362, 'val_chars': 4050551}


Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

supreme_court {'file_MB': 1322.567168, 'docs_scanned': 55027, 'train_chars': 1225015909, 'val_chars': 3120266}


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

tribunals {'file_MB': 6483.962193, 'docs_scanned': 773801, 'train_chars': 875015523, 'val_chars': 4003830}


train docs: 501734 val docs: 2297


## 3. Tokenizer: byte-level BPE (A1 algorithm, heap-based incremental merges)

In [3]:
PAT = re2.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+""")

def bytes_to_unicode():
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)); cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, [chr(c) for c in cs]))
B2U = bytes_to_unicode(); U2B = {v: k for k, v in B2U.items()}

def merge_word(w, a, b, new):
    out, i, n = [], 0, len(w)
    while i < n:
        if i < n - 1 and w[i] == a and w[i + 1] == b: out.append(new); i += 2
        else: out.append(w[i]); i += 1
    return out

class ByteBPE:
    SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
    def __init__(self, vocab_size=16384): self.vocab_size = vocab_size; self.merges = []; self.itos = []; self.cache = {}
    def fit(self, word_counts, max_unique=300_000):
        items = word_counts.most_common(max_unique)
        words = [[B2U[b] for b in w.encode("utf-8")] for w, _ in items]; freqs = [f for _, f in items]
        self.itos = self.SPECIALS + [B2U[b] for b in range(256)]; seen = set(self.itos)
        pc = defaultdict(int); where = defaultdict(set)
        for i, w in enumerate(words):
            for q in zip(w, w[1:]): pc[q] += freqs[i]; where[q].add(i)
        heap = [(-c, q) for q, c in pc.items()]; heapq.heapify(heap)
        n_merges = self.vocab_size - len(self.itos)
        while len(self.merges) < n_merges and heap:
            negc, p = heapq.heappop(heap); c = pc.get(p, 0)
            if c <= 0: continue
            if c != -negc: heapq.heappush(heap, (-c, p)); continue
            a, b = p; new = a + b; self.merges.append([a, b])
            if new not in seen: seen.add(new); self.itos.append(new)
            touched = set()
            for i in list(where[p]):
                w = words[i]; nw = merge_word(w, a, b, new)
                if len(nw) == len(w): continue
                f = freqs[i]
                for q in zip(w, w[1:]): pc[q] -= f
                for q in zip(nw, nw[1:]): pc[q] += f; where[q].add(i); touched.add(q)
                words[i] = nw
            where.pop(p, None); pc.pop(p, None)
            for q in touched:
                if pc[q] > 0: heapq.heappush(heap, (-pc[q], q))
            if len(self.merges) % 2000 == 0: print("merges", len(self.merges), flush=True)
        self.finalize(); return self
    def finalize(self):
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.rank = {tuple(m): i for i, m in enumerate(self.merges)}
        self.tbl = [b"" if i < 4 else bytes(U2B[ch] for ch in s) for i, s in enumerate(self.itos)]
    def save(self, path):
        json.dump({"merges": self.merges, "itos": self.itos, "vocab_size": len(self.itos)}, open(path, "w", encoding="utf-8"), ensure_ascii=False)
    @classmethod
    def load(cls, path):
        d = json.load(open(path, encoding="utf-8")); t = cls(d["vocab_size"]); t.merges = d["merges"]; t.itos = d["itos"]; t.finalize(); return t
    def word_ids(self, w):
        r = self.cache.get(w)
        if r is None:
            s = [B2U[b] for b in w.encode("utf-8")]
            while len(s) > 1:
                best, br = None, 1 << 60
                for q in zip(s, s[1:]):
                    rk = self.rank.get(q)
                    if rk is not None and rk < br: br, best = rk, q
                if best is None: break
                s = merge_word(s, best[0], best[1], best[0] + best[1])
            r = [self.stoi[x] for x in s]; self.cache[w] = r
        return r
    def encode(self, text):
        out = []
        for w in PAT.findall(text): out.extend(self.word_ids(w))
        return out
    def decode(self, ids): return b"".join(self.tbl[i] for i in ids if i < len(self.tbl)).decode("utf-8", "replace")

if os.path.exists(TOK_PATH):
    TOK = ByteBPE.load(TOK_PATH)
else:
    t0 = time.time(); idx = list(range(len(train_docs))); rng.shuffle(idx); wc = Counter(); ch = 0
    for i in idx:
        wc.update(PAT.findall(train_docs[i])); ch += len(train_docs[i])
        if ch >= TOK_CHARS: break
    print("unique pretokens:", len(wc), f"({time.time()-t0:.0f}s)")
    TOK = ByteBPE(VOCAB).fit(wc); TOK.save(TOK_PATH); del wc
print("vocab:", len(TOK.itos), "merges:", len(TOK.merges)); EOS = 3
print(TOK.decode(TOK.encode("The Hon'ble Supreme Court held under Section 138 of the N.I. Act, 1881 that ₹5,00,000 is payable.")))

unique pretokens: 217329 (7s)


merges 2000


merges 4000


merges 6000


merges 8000


merges 10000


merges 12000


merges 14000


merges 16000


vocab: 16384 merges: 16124
The Hon'ble Supreme Court held under Section 138 of the N.I. Act, 1881 that ₹5,00,000 is payable.


## 4. Encode corpus (multiprocessing, before JAX is initialised)

In [4]:
import multiprocessing as mp
DOCS = train_docs
def _enc_range(a):
    s, e = a; outs, lens = [], []
    for t in DOCS[s:e]:
        ids = TOK.encode(t); ids.append(EOS); outs.append(np.asarray(ids, dtype=np.uint16)); lens.append(len(ids))
    return np.concatenate(outs), np.asarray(lens)

if os.path.exists(TRAIN_NPY) and os.path.exists(VAL_NPY):
    train_tokens = np.load(TRAIN_NPY); val_tokens = np.load(VAL_NPY)
else:
    t0 = time.time(); CH = 400
    ranges = [(s, min(s + CH, len(DOCS))) for s in range(0, len(DOCS), CH)]
    with mp.get_context("fork").Pool(min(os.cpu_count() or 1, 32)) as pool:
        res = list(pool.imap(_enc_range, ranges))
    train_tokens = np.concatenate([r[0] for r in res]); doc_lens = np.concatenate([r[1] for r in res])
    per_src = defaultdict(int)
    for s, n in zip(train_src, doc_lens): per_src[s] += int(n)
    print("tokens per source:", dict(per_src), f"| total {len(train_tokens)/1e6:.1f}M | chars/token {sum(len(d) for d in DOCS)/len(train_tokens):.2f} | {time.time()-t0:.0f}s")
    val_tokens = np.concatenate([np.asarray(TOK.encode(t) + [EOS], dtype=np.uint16) for _, t in val_docs])
    np.save(TRAIN_NPY, train_tokens); np.save(VAL_NPY, val_tokens)
print("train tokens:", len(train_tokens), "val tokens:", len(val_tokens))
DOCS = None; train_docs = train_src = None; gc.collect()

# validation prompts mimicking the test format: 512-word prompt -> next 128 words
vr = random.Random(SEED + 1); cand = [t.split() for _, t in val_docs]; cand = [w for w in cand if len(w) >= 512 + TARGET_WORDS]
vr.shuffle(cand); val_prompts, val_refs = [], []
for w in cand[:N_VAL_PROMPTS]:
    s = vr.randint(0, len(w) - 512 - TARGET_WORDS)
    val_prompts.append(" ".join(w[s:s + 512])); val_refs.append(" ".join(w[s + 512:s + 512 + TARGET_WORDS]))
print("val prompts:", len(val_prompts))

tokens per source: {'acts': 38814725, 'high_courts': 286133362, 'supreme_court': 264694033, 'tribunals': 222234798} | total 811.9M | chars/token 4.31 | 19s


train tokens: 811876918 val tokens: 3196452


val prompts: 96


## 5. Model: hand-written JAX decoder family (GPT-2-style, LLaMA-style, and our hybrid variants)

In [5]:
import jax, jax.numpy as jnp
from jax import lax
from functools import partial
NDEV = jax.local_device_count(); GB = NDEV * PCB; BF = jnp.bfloat16; F32 = jnp.float32
print("devices:", jax.devices()[:2], "x", NDEV)

# Architectural variants. "ours_*" = our design: GQA + RoPE + RMSNorm + SwiGLU + QK-norm,
# alternating local(window 256)/global attention (2 local : 1 global), optional learned residual gates.
VARIANTS = {
  "gpt2_style":       dict(norm="ln",  mlp="gelu",   pos="learned", kvh=12, qknorm=False, hybrid=False, gate=False, ff=3072),
  "llama_style":      dict(norm="rms", mlp="swiglu", pos="rope",    kvh=4,  qknorm=False, hybrid=False, gate=False, ff=2048),
  "ours_hybrid":      dict(norm="rms", mlp="swiglu", pos="rope",    kvh=4,  qknorm=True,  hybrid=True,  gate=False, ff=2048),
  "ours_hybrid_gate": dict(norm="rms", mlp="swiglu", pos="rope",    kvh=4,  qknorm=True,  hybrid=True,  gate=True,  ff=2048),
  # Attention-free: every block is a gated linear recurrence (R). No positional encoding needed.
  "ours_glru":        dict(norm="rms", mlp="swiglu", pos="rope",    kvh=4,  qknorm=True,  hybrid=False, gate=False, ff=2048, kinds="R" * 12),
  # Hybrid: two recurrent blocks then one global-attention block (R R A) x 4
  "ours_glru_attn":   dict(norm="rms", mlp="swiglu", pos="rope",    kvh=4,  qknorm=True,  hybrid=False, gate=False, ff=2048, kinds="RRA" * 4),
}
SSM_K, DR = 4, D        # depthwise causal-conv width and recurrent width of the gated linear recurrence
MATS = {"wq", "wk", "wv", "wo", "w1", "w2", "w3", "wx", "wg", "wa"}   # weight-decayed matrices
LOCAL_WINDOW = 256

def make_windows(c): return np.array([LOCAL_WINDOW if (c["hybrid"] and i % 3 != 2) else 1 << 30 for i in range(L)], np.int32)

def init_params_blocks(key, c):
    ks = jax.random.split(key, len(c["kinds"]) + 2); KV, ff = c["kvh"], c["ff"]
    nrm = lambda k, s, sd=0.02: jax.random.normal(k, s, F32) * sd
    so = 0.02 / math.sqrt(2 * L); a0 = jnp.linspace(0.9, 0.999, DR); blocks = []
    for i, kind in enumerate(c["kinds"]):
        kk = jax.random.split(ks[i], 10)
        lp = {"n1": jnp.ones((D,)), "n2": jnp.ones((D,)), "w1": nrm(kk[5], (D, ff)), "w3": nrm(kk[6], (D, ff)), "w2": nrm(kk[7], (ff, D), so)}
        if kind == "A":
            lp.update(wq=nrm(kk[0], (D, H * HD)), wk=nrm(kk[1], (D, KV * HD)), wv=nrm(kk[2], (D, KV * HD)), wo=nrm(kk[3], (H * HD, D), so), qn=jnp.ones((HD,)), kn=jnp.ones((HD,)))
        else:   # gated linear recurrence block; forget-gate bias spreads time-scales from ~10 to ~1000 tokens
            lp.update(wx=nrm(kk[0], (D, DR)), wg=nrm(kk[1], (D, DR)), wa=nrm(kk[2], (D, DR)), ab=jnp.log(a0 / (1 - a0)),
                      conv=jnp.zeros((SSM_K, DR)).at[SSM_K - 1].set(1.0) + nrm(kk[3], (SSM_K, DR)), wo=nrm(kk[4], (DR, D), so))
        blocks.append(lp)
    return {"emb": nrm(ks[-1], (V, D)), "blocks": blocks, "nf": jnp.ones((D,))}

def init_params(key, c):
    if c.get("kinds"): return init_params_blocks(key, c)
    ks = jax.random.split(key, 12); KV, ff = c["kvh"], c["ff"]
    nrm = lambda k, s, sd=0.02: jax.random.normal(k, s, F32) * sd
    so = 0.02 / math.sqrt(2 * L)
    lp = {"wq": nrm(ks[0], (L, D, H * HD)), "wk": nrm(ks[1], (L, D, KV * HD)), "wv": nrm(ks[2], (L, D, KV * HD)),
          "wo": nrm(ks[3], (L, H * HD, D), so), "n1": jnp.ones((L, D)), "n2": jnp.ones((L, D))}
    if c["mlp"] == "swiglu":
        lp.update(w1=nrm(ks[4], (L, D, ff)), w3=nrm(ks[5], (L, D, ff)), w2=nrm(ks[6], (L, ff, D), so))
    else:
        lp.update(w1=nrm(ks[4], (L, D, ff)), b1=jnp.zeros((L, ff)), w2=nrm(ks[6], (L, ff, D), so), b2=jnp.zeros((L, D)))
    if c["norm"] == "ln": lp.update(n1b=jnp.zeros((L, D)), n2b=jnp.zeros((L, D)))
    if c["qknorm"]: lp.update(qn=jnp.ones((L, HD)), kn=jnp.ones((L, HD)))
    if c["gate"]: lp["g"] = jnp.ones((L, 2))
    p = {"emb": nrm(ks[7], (V, D)), "layers": lp, "nf": jnp.ones((D,))}
    if c["norm"] == "ln": p["nfb"] = jnp.zeros((D,))
    if c["pos"] == "learned": p["pos"] = nrm(ks[8], (CTX, D))
    return p

def n_params(p): return int(sum(np.prod(x.shape) for x in jax.tree_util.tree_leaves(p)))

_inv = 1.0 / (10000.0 ** (np.arange(0, HD, 2) / HD)); _t = np.arange(CTX)[:, None] * _inv[None]
ROPE_COS, ROPE_SIN = jnp.asarray(np.cos(_t), F32), jnp.asarray(np.sin(_t), F32)

def apply_rope(x, cos, sin):
    x1, x2 = x[..., :HD // 2].astype(F32), x[..., HD // 2:].astype(F32); cos, sin = cos[:, :, None, :], sin[:, :, None, :]
    return jnp.concatenate([x1 * cos - x2 * sin, x2 * cos + x1 * sin], -1).astype(BF)

def norm_f(c, x, w, b=None):
    x = x.astype(F32)
    if c["norm"] == "ln":
        mu = x.mean(-1, keepdims=True); y = (x - mu) * lax.rsqrt(((x - mu) ** 2).mean(-1, keepdims=True) + 1e-5) * w + b
    else:
        y = x * lax.rsqrt((x ** 2).mean(-1, keepdims=True) + 1e-6) * w
    return y.astype(BF)

def rms_h(x, w):
    x = x.astype(F32); return (x * lax.rsqrt((x ** 2).mean(-1, keepdims=True) + 1e-6) * w).astype(BF)

def layer(c, x, lp, win, cos, sin, qpos, kpos, kval, kc=None, vc=None, widx=None):
    B, T, _ = x.shape; KV = c["kvh"]; R = H // KV
    h = norm_f(c, x, lp["n1"], lp.get("n1b"))
    q = (h @ lp["wq"].astype(BF)).reshape(B, T, H, HD)
    k = (h @ lp["wk"].astype(BF)).reshape(B, T, KV, HD)
    v = (h @ lp["wv"].astype(BF)).reshape(B, T, KV, HD)
    if c["qknorm"]: q, k = rms_h(q, lp["qn"]), rms_h(k, lp["kn"])
    if c["pos"] == "rope": q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
    if kc is not None:
        z = jnp.int32(0)
        kc = lax.dynamic_update_slice(kc, k, (z, widx, z, z)); vc = lax.dynamic_update_slice(vc, v, (z, widx, z, z)); K, Vv = kc, vc
    else: K, Vv = k, v
    q = q.reshape(B, T, KV, R, HD)
    s = jnp.einsum("btgrd,bsgd->bgrts", q, K, preferred_element_type=F32) * (HD ** -0.5)
    m = (kpos[:, None, :] <= qpos[:, :, None]) & ((qpos[:, :, None] - kpos[:, None, :]) < win) & kval[:, None, :]
    a = jax.nn.softmax(jnp.where(m[:, None, None], s, -1e30), -1).astype(BF)
    o = jnp.einsum("bgrts,bsgd->btgrd", a, Vv).reshape(B, T, H * HD) @ lp["wo"].astype(BF)
    g1, g2 = (lp["g"][0], lp["g"][1]) if c["gate"] else (1.0, 1.0)
    x = x + (g1 * o).astype(F32)
    h = norm_f(c, x, lp["n2"], lp.get("n2b"))
    if c["mlp"] == "swiglu": mo = (jax.nn.silu(h @ lp["w1"].astype(BF)) * (h @ lp["w3"].astype(BF))) @ lp["w2"].astype(BF)
    else: mo = jax.nn.gelu(h @ lp["w1"].astype(BF) + lp["b1"].astype(BF)) @ lp["w2"].astype(BF) + lp["b2"].astype(BF)
    x = x + (g2 * mo).astype(F32)
    return x, kc, vc

def _comb(l, r): return (l[0] * r[0], r[0] * l[1] + r[1])      # associative operator of  s_t = a_t * s_{t-1} + b_t

def rec_layer(c, x, lp, valid, state):
    """Gated linear-recurrent block: causal depthwise conv -> input-dependent forget gate a_t -> s_t = a_t*s_{t-1} + (1-a_t)*u_t
    -> output gate silu(Wg h) -> Wo, followed by a SwiGLU MLP. Training/prefill use a parallel associative scan; decoding is O(1)/token."""
    B, T, _ = x.shape; K = lp["conv"].shape[0]
    h = norm_f(c, x, lp["n1"])
    xv = (h @ lp["wx"].astype(BF)).astype(F32)
    if valid is not None: xv = xv * valid[..., None]                 # left-padding contributes nothing
    buf = state["buf"] if state is not None else jnp.zeros((B, K - 1, xv.shape[-1]), F32)
    xp = jnp.concatenate([buf, xv], 1)
    u = sum(xp[:, j:j + T] * lp["conv"][j] for j in range(K))
    a = jax.nn.sigmoid((h @ lp["wa"].astype(BF)).astype(F32) + lp["ab"]); b = (1.0 - a) * u
    if valid is not None: a = jnp.where(valid[..., None] > 0, a, 1.0); b = b * valid[..., None]
    if T > 1: _, s = lax.associative_scan(_comb, (a, b), axis=1)
    else: s = a * state["s"][:, None] + b
    y = ((s * jax.nn.silu((h @ lp["wg"].astype(BF)).astype(F32))).astype(BF)) @ lp["wo"].astype(BF)
    x = x + y.astype(F32)
    h2 = norm_f(c, x, lp["n2"])
    x = x + ((jax.nn.silu(h2 @ lp["w1"].astype(BF)) * (h2 @ lp["w3"].astype(BF))) @ lp["w2"].astype(BF)).astype(F32)
    return x, (dict(s=s[:, -1], buf=xp[:, -(K - 1):]) if state is not None else None)

def run_blocks(c, params, x, valid, qpos, kpos, kval, cos, sin, caches, widx):
    new = []
    for i, (k, lp) in enumerate(zip(c["kinds"], params["blocks"])):
        st = caches[i]
        if k == "A":
            x, kc, vc = layer(c, x, lp, jnp.int32(1 << 30), cos, sin, qpos, kpos, kval, st[0], st[1], widx); new.append((kc, vc))
        else:
            x, ns = rec_layer(c, x, lp, valid, st); new.append(ns)
    return x, new

def forward_train_blocks(c, params, tokens):
    T = tokens.shape[1]; x = params["emb"][tokens]; pos = jnp.arange(T)[None]; cos, sin = ROPE_COS[pos], ROPE_SIN[pos]; kval = jnp.ones((1, T), bool)
    fa = jax.checkpoint(lambda x, lp: layer(c, x, lp, jnp.int32(1 << 30), cos, sin, pos, pos, kval)[0])
    fr = jax.checkpoint(lambda x, lp: rec_layer(c, x, lp, None, None)[0])
    for k, lp in zip(c["kinds"], params["blocks"]): x = (fa if k == "A" else fr)(x, lp)
    return norm_f(c, x, params["nf"]) @ params["emb"].astype(BF).T

def forward_train(c, win, params, tokens):
    if c.get("kinds"): return forward_train_blocks(c, params, tokens)
    T = tokens.shape[1]; x = params["emb"][tokens]
    if c["pos"] == "learned": x = x + params["pos"][:T]
    pos = jnp.arange(T)[None]; cos, sin = ROPE_COS[pos], ROPE_SIN[pos]; kval = jnp.ones((1, T), bool)
    def body(x, xs):
        lp, w = xs; x, _, _ = layer(c, x, lp, w, cos, sin, pos, pos, kval); return x, None
    x, _ = lax.scan(jax.checkpoint(body, prevent_cse=False), x, (params["layers"], jnp.asarray(win)))
    x = norm_f(c, x, params["nf"], params.get("nfb"))
    return x @ params["emb"].astype(BF).T

def xent(logits, y):
    logits = logits.astype(F32); lse = jax.nn.logsumexp(logits, -1)
    return (lse - jnp.take_along_axis(logits, y[..., None], -1)[..., 0]).mean()

for _n, _c in VARIANTS.items():
    _p = init_params(jax.random.PRNGKey(0), _c); _k = n_params(_p); assert 50e6 <= _k <= 300e6, _k
    print(f"{_n:18s} {_k/1e6:6.1f}M params (within 50M-300M)")
del _p

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


E0000 00:00:1790995117.030509      73 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0)] x 8


gpt2_style           98.6M params (within 50M-300M)


llama_style          88.1M params (within 50M-300M)
ours_hybrid          88.1M params (within 50M-300M)
ours_hybrid_gate     88.1M params (within 50M-300M)


ours_glru            97.6M params (within 50M-300M)


ours_glru_attn       94.4M params (within 50M-300M)


## 6. Training utilities (pmap data-parallel, hand-written AdamW, cosine schedule, time-budget aware)

In [6]:
class Batcher:
    def __init__(s, toks, seed): s.t = toks; s.n = (len(toks) - 1) // CTX; s.r = np.random.RandomState(seed); s.perm = s.r.permutation(s.n); s.ptr = 0
    def next(s):
        if s.ptr + GB > s.n: s.perm = s.r.permutation(s.n); s.ptr = 0
        i = s.perm[s.ptr:s.ptr + GB]; s.ptr += GB
        return s.t[i[:, None] * CTX + np.arange(CTX + 1)[None]].astype(np.int32).reshape(NDEV, PCB, CTX + 1)

_nb = (len(val_tokens) - 1) // CTX
VAL = val_tokens[(np.arange(_nb)[:, None] * CTX + np.arange(CTX + 1)[None])].astype(np.int32)
VAL_BATCHES = min(len(VAL) // GB, 24); assert VAL_BATCHES > 0
VAL = VAL[:VAL_BATCHES * GB].reshape(VAL_BATCHES, NDEV, PCB, CTX + 1)

def rep(tree):
    try: return jax.device_put_replicated(tree, jax.local_devices())
    except Exception:
        return jax.pmap(lambda t: t)(jax.tree_util.tree_map(lambda x: np.broadcast_to(np.asarray(x), (NDEV,) + np.shape(x)), tree))
def unrep(tree): return jax.tree_util.tree_map(lambda x: np.asarray(x[0]), tree)

def make_fns(c, win):
    tmap = jax.tree_util.tree_map
    def loss_fn(p, b): return xent(forward_train(c, win, p, b[:, :-1]), b[:, 1:])
    def lr_at(step, total, warm):
        w = step / warm; pr = jnp.clip((step - warm) / jnp.maximum(total - warm, 1.0), 0.0, 1.0)
        return PEAK_LR * jnp.where(step < warm, w, MIN_LR_FRAC + (1 - MIN_LR_FRAC) * 0.5 * (1 + jnp.cos(jnp.pi * pr)))
    @partial(jax.pmap, axis_name="d", in_axes=(0, 0, 0, 0, None, None, None), donate_argnums=(0, 1, 2))
    def train_step(params, m, v, batch, step, total, warm):
        loss, g = jax.value_and_grad(loss_fn)(params, batch)
        g = lax.pmean(g, "d"); loss = lax.pmean(loss, "d")
        gn = jnp.sqrt(sum(jnp.sum(x.astype(F32) ** 2) for x in jax.tree_util.tree_leaves(g)))
        sc = jnp.minimum(1.0, CLIP / (gn + 1e-6)); lr = lr_at(step, total, warm); t = step + 1.0
        nm = tmap(lambda m_, g_: B1 * m_ + (1 - B1) * g_ * sc, m, g)
        nv = tmap(lambda v_, g_: B2 * v_ + (1 - B2) * (g_ * sc) ** 2, v, g)
        npar = tmap(lambda p, m_, v_, dm: p - lr * ((m_ / (1 - B1 ** t)) / (jnp.sqrt(v_ / (1 - B2 ** t)) + EPS) + WD * dm * p), params, nm, nv, DECAY)
        return npar, nm, nv, loss, gn
    @partial(jax.pmap, axis_name="d")
    def eval_step(params, batch): return lax.pmean(loss_fn(params, batch), "d")
    return train_step, eval_step

def evaluate(eval_step, params):
    return float(np.mean([float(eval_step(params, VAL[i])[0]) for i in range(VAL_BATCHES)]))

def run_training(name, c, fixed_steps=None, budget_sec=None, warmup=200, ckpt=None, eval_every=500, log_every=50, ckpt_every_sec=1500):
    global DECAY
    win = make_windows(c)
    p0 = init_params(jax.random.PRNGKey(SEED), c); n = n_params(p0); assert 50e6 <= n <= 300e6
    DECAY = jax.tree_util.tree_map_with_path(lambda path, x: 1.0 if getattr(path[-1], "key", None) in MATS else 0.0, p0)
    train_step, eval_step = make_fns(c, win)
    step, total, hist = 0, (fixed_steps or 10 ** 9), []
    if ckpt and os.path.exists(ckpt):
        d = pickle.load(open(ckpt, "rb")); params, m, v = rep(d["params"]), rep(d["m"]), rep(d["v"]); step, total, hist = d["step"], d["total"], d["hist"]
        print(f"resumed {name} at step {step}/{total}")
    else:
        params = rep(p0); m = rep(jax.tree_util.tree_map(jnp.zeros_like, p0)); v = rep(jax.tree_util.tree_map(jnp.zeros_like, p0))
    del p0
    bt = Batcher(train_tokens, SEED + step); run_t0 = time.time(); last_ck = time.time(); tb = None; losses = []
    need_bench = budget_sec is not None and total == 10 ** 9
    while step < total:
        params, m, v, loss, gn = train_step(params, m, v, bt.next(), jnp.float32(step), jnp.float32(total), jnp.float32(warmup))
        step += 1
        if need_bench and step == 6: loss.block_until_ready(); tb = time.time()
        if need_bench and step == 26:
            loss.block_until_ready(); rate = (time.time() - tb) / 20.0
            total = step + int(0.96 * (budget_sec - (time.time() - run_t0)) / rate); need_bench = False
            print(f"[{name}] {rate:.3f}s/step, {GB*CTX/rate:,.0f} tok/s -> total_steps={total} (~{total*GB*CTX/1e6:.0f}M tokens)")
        if step % log_every == 0:
            lv = float(loss[0]); hist.append(dict(step=step, tokens=step * GB * CTX, train_loss=lv, val_loss=None, time=time.time() - run_t0))
            if step % (log_every * 4) == 0: print(f"[{name}] step {step}/{total} loss {lv:.4f} gn {float(gn[0]):.2f} | {(time.time()-run_t0)/60:.1f} min", flush=True)
        if step % eval_every == 0 or step == total:
            vl = evaluate(eval_step, params); hist.append(dict(step=step, tokens=step * GB * CTX, train_loss=None, val_loss=vl, time=time.time() - run_t0))
            print(f"[{name}] step {step} VAL loss {vl:.4f} ppl {math.exp(vl):.2f}", flush=True)
        if ckpt and time.time() - last_ck > ckpt_every_sec:
            pickle.dump(dict(params=unrep(params), m=unrep(m), v=unrep(v), step=step, total=total, hist=hist), open(ckpt + ".tmp", "wb")); os.replace(ckpt + ".tmp", ckpt); last_ck = time.time()
        if time.time() - T0 > TOTAL_SEC - RESERVE_SEC: print("time guard hit, stopping early"); break
    vl = evaluate(eval_step, params); secs = time.time() - run_t0
    return params, dict(name=name, params_M=n / 1e6, steps=step, tokens=step * GB * CTX, val_loss=vl, val_ppl=math.exp(vl), seconds=secs, tok_per_s=step * GB * CTX / secs, hist=hist)

print("utilities ready")

utilities ready


## 7. Architecture ablation (equal tokens per variant, same data order and seed)

In [7]:
ABL_PATH = f"{WORK}/ablation_results.json"
if os.path.exists(ABL_PATH): ABL = json.load(open(ABL_PATH))
else:
    ABL = {}
    for name, c in VARIANTS.items():
        p, info = run_training("abl_" + name, c, fixed_steps=ABL_STEPS, warmup=30, eval_every=100, log_every=25)
        info.pop("hist"); ABL[name] = info; del p; gc.collect()
        print(f">>> {name}: val_loss {info['val_loss']:.4f} | {info['tok_per_s']:,.0f} tok/s | {info['params_M']:.1f}M")
        json.dump(ABL, open(ABL_PATH, "w"), indent=1)
print(f"{'variant':20s}{'paramsM':>9s}{'val_loss':>10s}{'val_ppl':>9s}{'tok/s':>10s}")
for k, r in ABL.items(): print(f"{k:20s}{r['params_M']:9.1f}{r['val_loss']:10.4f}{r['val_ppl']:9.2f}{r['tok_per_s']:10,.0f}")
BEST = min([k for k in ABL if k.startswith("ours_")], key=lambda k: ABL[k]["val_loss"])
print("selected for final run:", BEST)

[abl_gpt2_style] step 100/300 loss 5.6410 gn 0.56 | 0.6 min


[abl_gpt2_style] step 100 VAL loss 5.6141 ppl 274.26


[abl_gpt2_style] step 200/300 loss 5.1012 gn 0.42 | 0.9 min


[abl_gpt2_style] step 200 VAL loss 5.0878 ppl 162.04


[abl_gpt2_style] step 300/300 loss 4.9392 gn 0.33 | 1.3 min


[abl_gpt2_style] step 300 VAL loss 4.9194 ppl 136.92


... (32 outputs left out)

[abl_ours_glru_attn] step 200 VAL loss 3.5795 ppl 35.86


[abl_ours_glru_attn] step 300/300 loss 3.3841 gn 0.45 | 9.4 min


[abl_ours_glru_attn] step 300 VAL loss 3.3367 ppl 28.13


>>> ours_glru_attn: val_loss 3.3367 | 43,422 tok/s | 94.4M
variant               paramsM  val_loss  val_ppl     tok/s
gpt2_style               98.6    4.9194   136.92   316,953
llama_style              88.1    3.9978    54.48   347,906
ours_hybrid              88.1    4.2278    68.57   336,059
ours_hybrid_gate         88.1    4.0426    56.97   328,702
ours_glru                97.6    3.2920    26.90    35,925
ours_glru_attn           94.4    3.3367    28.13    43,422
selected for final run: ours_glru


## 8. Final training (time-budgeted, checkpointed, resumable)

In [8]:
elapsed = time.time() - T0
budget = min(FINAL_MAX_MIN * 60, TOTAL_SEC - elapsed - RESERVE_SEC)
print(f"elapsed {elapsed/60:.0f} min -> final training budget {budget/60:.0f} min")
FINAL_CKPT = f"{WORK}/final_ckpt.pkl"
C = VARIANTS[BEST]
params_rep, FINAL = run_training("final_" + BEST, C, budget_sec=budget, warmup=300, ckpt=FINAL_CKPT, eval_every=500, log_every=50)
pickle.dump(dict(params=unrep(params_rep), cfg=C, variant=BEST), open(f"{WORK}/final_params.pkl", "wb"))
json.dump({k: v for k, v in FINAL.items() if k != "hist"}, open(f"{WORK}/final_summary.json", "w"), indent=1)
print({k: v for k, v in FINAL.items() if k != "hist"})

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
h = FINAL["hist"]; tr = [(x["tokens"] / 1e6, x["train_loss"]) for x in h if x["train_loss"] is not None]; va = [(x["tokens"] / 1e6, x["val_loss"]) for x in h if x["val_loss"] is not None]
plt.figure(figsize=(9, 5)); plt.plot(*zip(*tr), label="train loss", alpha=.7); plt.plot(*zip(*va), "o-", label="val loss")
plt.xlabel("tokens (M)"); plt.ylabel("loss"); plt.grid(True); plt.legend(); plt.title(f"Phase 1 — {BEST}"); plt.savefig(f"{WORK}/loss_curve.png", dpi=200, bbox_inches="tight"); plt.show()

elapsed 35 min -> final training budget 330 min


[final_ours_glru] 0.108s/step, 755,387 tok/s -> total_steps=169732 (~13904M tokens)


[final_ours_glru] step 200/169732 loss 3.9867 gn 0.76 | 10.8 min


[final_ours_glru] step 400/169732 loss 3.2131 gn 0.47 | 11.2 min


[final_ours_glru] step 500 VAL loss 3.1234 ppl 22.72


[final_ours_glru] step 600/169732 loss 3.1532 gn 0.37 | 11.7 min


... (1,181 outputs left out)

[final_ours_glru] step 169500 VAL loss 1.7075 ppl 5.52


[final_ours_glru] step 169600/169732 loss 1.5749 gn 0.46 | 321.6 min


[final_ours_glru] step 169732 VAL loss 1.7047 ppl 5.50


{'name': 'final_ours_glru', 'params_M': 97.582848, 'steps': 169732, 'tokens': 13904445440, 'val_loss': 1.7047394663095474, 'val_ppl': 5.4999525575957895, 'seconds': 19311.33978843689, 'tok_per_s': 720014.5402819543}


## 9. Generation (KV-cache, pmap across cores): decoding sweep on held-out prompts

In [9]:
def make_gen(c, win):
    KV = c["kvh"]; winj = jnp.asarray(win); kinds = c.get("kinds")
    def init_cache(b):
        if kinds:
            return [(jnp.zeros((b, CTX, KV, HD), BF), jnp.zeros((b, CTX, KV, HD), BF)) if k == "A" else
                    dict(s=jnp.zeros((b, DR), F32), buf=jnp.zeros((b, SSM_K - 1, DR), F32)) for k in kinds]
        kc = jnp.zeros((L, b, CTX, KV, HD), BF); return (kc, jnp.zeros_like(kc))
    def fwd(params, tok, qpos, kpos, kval, valid, cache, widx):
        qp = jnp.maximum(qpos, 0); x = params["emb"][tok]
        if c["pos"] == "learned" and not kinds: x = x + params["pos"][qp]
        cos, sin = ROPE_COS[qp], ROPE_SIN[qp]
        if kinds:
            x, cache = run_blocks(c, params, x, valid, qpos, kpos, kval, cos, sin, cache, widx)
        else:
            def body(x, xs):
                lp, w, k_, v_ = xs; x, k_, v_ = layer(c, x, lp, w, cos, sin, qpos, kpos, kval, k_, v_, widx); return x, (k_, v_)
            x, (kc, vc) = lax.scan(body, x, (params["layers"], winj, cache[0], cache[1])); cache = (kc, vc)
        x = norm_f(c, x[:, -1:], params["nf"], params.get("nfb"))
        return (x @ params["emb"].astype(BF).T)[:, 0].astype(F32), cache
    @partial(jax.pmap, in_axes=(0, 0, 0, None, None, None, 0))
    def gen(params, prompt, pad, temp, topp, rp, key):
        b = prompt.shape[0]; cache = init_cache(b)
        slots = jnp.arange(CTX); kpos = slots[None] - pad[:, None]; kval = slots[None] >= pad[:, None]
        valid = (slots[None, :PROMPT_MAX] >= pad[:, None]).astype(F32)
        logits, cache = fwd(params, prompt, jnp.arange(PROMPT_MAX)[None] - pad[:, None], kpos, kval, valid, cache, jnp.int32(0))
        def step(carry, i):
            logits, cache, seen, key = carry; key, sk = jax.random.split(key)
            lg = logits.at[:, :4].set(-1e30)
            lg = jnp.where(seen, jnp.where(lg > 0, lg / rp, lg * rp), lg)
            greedy = jnp.argmax(lg, -1)
            vals, idx = lax.top_k(lg / jnp.maximum(temp, 1e-4), 64); pr = jax.nn.softmax(vals, -1)
            vals = jnp.where((jnp.cumsum(pr, -1) - pr) < topp, vals, -1e30)
            samp = jnp.take_along_axis(idx, jax.random.categorical(sk, vals, -1)[:, None], -1)[:, 0]
            tok = jnp.where(temp <= 0, greedy, samp); seen = seen.at[jnp.arange(b), tok].set(True)
            slot = PROMPT_MAX + i
            logits, cache = fwd(params, tok[:, None], (slot - pad)[:, None], kpos, kval, None, cache, slot)
            return (logits, cache, seen, key), tok
        _, toks = lax.scan(step, (logits, cache, jnp.zeros((b, V), bool), key), jnp.arange(GEN_TOK))
        return toks.T
    return gen

GEN = make_gen(C, make_windows(C))

def generate(prompts, temp=0.0, topp=1.0, rp=1.0, bs=8, seed=0):
    ids_all = [TOK.encode(p)[-PROMPT_MAX:] for p in prompts]; per = NDEV * bs; out = []
    for s in range(0, len(ids_all), per):
        chunk = ids_all[s:s + per]; real = len(chunk); chunk = chunk + [chunk[-1]] * (per - real)
        arr = np.zeros((per, PROMPT_MAX), np.int32); pad = np.zeros(per, np.int32)
        for i, ids in enumerate(chunk): pl = PROMPT_MAX - len(ids); arr[i, pl:] = ids; pad[i] = pl
        keys = jax.random.split(jax.random.PRNGKey(seed + s), NDEV)
        res = np.asarray(GEN(params_rep, arr.reshape(NDEV, bs, -1), pad.reshape(NDEV, bs), jnp.float32(temp), jnp.float32(topp), jnp.float32(rp), keys)).reshape(per, -1)[:real]
        for r in res:
            w = TOK.decode([int(t) for t in r]).split()[:TARGET_WORDS]; out.append(" ".join(w) if w else "the")
    return out

# ---- local metrics (same definitions as evaluate_phase1.py) ----
def lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b): cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]
def rouge_l(p, r):
    p, r = p.split(), r.split(); l = lcs(p, r)
    return 0.0 if l == 0 else 2 * (l / len(p)) * (l / len(r)) / ((l / len(p)) + (l / len(r)))
def bleu4(preds, refs):
    from collections import Counter as C_; cl, tot, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        p, r = p.split(), r.split(); pl += len(p); rl += len(r)
        for n in range(1, 5):
            pn = C_(tuple(p[i:i + n]) for i in range(len(p) - n + 1)); rn = C_(tuple(r[i:i + n]) for i in range(len(r) - n + 1)); tot[n - 1] += sum(pn.values())
            cl[n - 1] += sum(min(c, rn.get(g, 0)) for g, c in pn.items())
    pr = [cl[i] / tot[i] if tot[i] else 0 for i in range(4)]
    return 0.0 if min(pr) <= 0 or pl == 0 else (1.0 if pl > rl else math.exp(1 - rl / pl)) * math.exp(sum(math.log(x) for x in pr) / 4)
SCORER = None
def score(preds, refs):
    global SCORER
    out = dict(rougeL=float(np.mean([rouge_l(p, r) for p, r in zip(preds, refs)])), bleu4=bleu4(preds, refs), bertscore=None)
    try:
        if SCORER is None:
            import bert_score; SCORER = bert_score.BERTScorer(lang="en", device="cpu", batch_size=8)
        out["bertscore"] = float(SCORER.score(preds, refs)[2].mean())
    except Exception as e: print("BERTScore unavailable:", e)
    return out

SWEEP = [(0.0, 1.0, 1.0), (0.0, 1.0, 1.1), (0.0, 1.0, 1.2), (0.3, 0.9, 1.1), (0.5, 0.9, 1.15)]   # (temperature, top_p, repetition penalty)
SWEEP_RES = []
for (t, tp, rp) in SWEEP:
    preds = generate(val_prompts, t, tp, rp); sc = score(preds, val_refs); SWEEP_RES.append(((t, tp, rp), sc))
    print(f"temp={t} top_p={tp} rep={rp} ->", {k: (round(v, 4) if v is not None else None) for k, v in sc.items()}, flush=True)
key_fn = lambda r: r[1]["bertscore"] if r[1]["bertscore"] is not None else r[1]["rougeL"]
BEST_DEC = max(SWEEP_RES, key=key_fn)[0]; print("best decoding:", BEST_DEC)
json.dump([[list(k), v] for k, v in SWEEP_RES], open(f"{WORK}/decoding_sweep.json", "w"), indent=1)
print("\nSAMPLE\nPROMPT TAIL:", " ".join(val_prompts[0].split()[-40:]), "\nGENERATED:", generate(val_prompts[:NDEV * 8], *BEST_DEC)[0], "\nREFERENCE:", val_refs[0])

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (2 outputs left out)

BERTScore unavailable: RobertaProcessing.__new__() got an unexpected keyword argument 'cls'
temp=0.0 top_p=1.0 rep=1.2 -> {'rougeL': 0.2828, 'bleu4': 0.2024, 'bertscore': None}


BERTScore unavailable: RobertaProcessing.__new__() got an unexpected keyword argument 'cls'
temp=0.3 top_p=0.9 rep=1.1 -> {'rougeL': 0.3197, 'bleu4': 0.2347, 'bertscore': None}


BERTScore unavailable: RobertaProcessing.__new__() got an unexpected keyword argument 'cls'
temp=0.5 top_p=0.9 rep=1.15 -> {'rougeL': 0.2925, 'bleu4': 0.2076, 'bertscore': None}


best decoding: (0.0, 1.0, 1.0)

SAMPLE
PROMPT TAIL: opinion clinches the matter. To that effect, the CAT's decision that the entry of departmental candidates to the cadre of Postal Assistant is by way of direct recruitment is unexceptionable. We consequently affirm the findings of the CAT in the 
GENERATED: impugned order." Hence, the SLP(C)No.8551/2016 filed by the Department was dismissed by the Hon'ble Apex Court. 14. As we.No.60629/2019 and 677/2019 were filed seeking to challenge the order of the Bangalore Coordinate Bench of this Tribunal dated 16.10.2016 in OA.No.1610/2016, the same was dismissed as withdrawn with liberty to file a review before the Hon'ble High Court of Karnataka in the order dated 13.08.2016 in WP.No.32524/2016. A S S U S S Aw ar i v. Sl.No.(it) By its order dated 20.07.2016 in WP.No.32524/2016 and the Hon'ble High Court on 17.11.2017 in WP.No.32524/2017 and the Hon'ble Apex Court vide order dated 08.08.2017 disposed off the said Writ Petition. As per the orde

## 10. Test predictions and Codabench zip

In [10]:
test_path = hf_hub_download(HF_REPO, test_file, repo_type="dataset", token=HF_TOKEN, local_dir=TMP)
test_data = json.load(open(test_path, encoding="utf-8")); ids = [str(k) for k in test_data]
print("test examples:", len(ids))
preds = generate([test_data[k]["text"] for k in ids], *BEST_DEC)
assert len(preds) == len(ids) and all(p.strip() for p in preds)
prediction = {k: {"text": p} for k, p in zip(ids, preds)}
PRED_PATH, ZIP_PATH = f"{WORK}/prediction.json", f"{WORK}/submission.zip"
json.dump(prediction, open(PRED_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z: z.write(PRED_PATH, arcname="prediction.json")
print("words per prediction (min/mean/max):", min(len(p.split()) for p in preds), np.mean([len(p.split()) for p in preds]), max(len(p.split()) for p in preds))
print("saved:", ZIP_PATH, f"| total notebook time {(time.time()-T0)/3600:.2f} h")

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

test examples: 1000


words per prediction (min/mean/max): 37 127.627 128
saved: /kaggle/working/cs781_phase1/submission.zip | total notebook time 5.98 h
